In [1]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import re

# ✅ run 폴더
run_dir = Path('..').resolve()

print("run_dir exists:", run_dir.exists(), run_dir)


run_dir exists: True /home/dibaeck/workspace/project_IR_sLM_MAS/runs/exp2_qwen2p5_policy_v0_smoke_20260331_120344


In [2]:
merged_path = run_dir / "merged_results.jsonl"
print("merged_path:", merged_path)
print("exists:", merged_path.exists())

merged_path: /home/dibaeck/workspace/project_IR_sLM_MAS/runs/exp2_qwen2p5_policy_v0_smoke_20260331_120344/merged_results.jsonl
exists: True


In [3]:
# JSONL 로더
rows = []
with open(merged_path, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        rows.append(json.loads(line))

print("num_rows:", len(rows))
rows[0].keys() if rows else None

num_rows: 100


dict_keys(['task_id', 'trial_id', 'attempt_index', 'model', 'prompt_hash', 'diff', 'edit_script', 'patch_lines_added', 'patch_lines_removed', 'files_changed', 'timestamp', 'seed', 'repo', 'base_commit', 'taxonomy_version', 'gen_elapsed_sec', 'context_used', 'context_num_files', 'repo_context_preview', 'edit_used', 'edit_parse_ok', 'edit_parse_reason', 'diff_export_ok', 'diff_export_reason', 'success', 'stage', 'error_type', 'signature', 'stdout', 'stderr', 'returncode', 'timeout', 'elapsed_sec', 'test_command', 'generated_diff', 'exception', 'policy_enabled', 'trigger_error_type', 'trigger_signature', 'policy_action', 'policy_params', 'final_selected', 'terminated_by_policy', 'instance_id', 'had_prediction', 'merged_from_pre', 'merged_from_harness', 'harness_log_roots', 'final_error_type', 'final_signature', 'final_stage', 'final_success', 'final_source'])

In [4]:
# DataFrame 변환
df = pd.DataFrame(rows)

print("shape:", df.shape)
display(df.head(3))

shape: (100, 54)


,task_id,trial_id,attempt_index,model,prompt_hash,diff,edit_script,patch_lines_added,patch_lines_removed,files_changed,...,had_prediction,merged_from_pre,merged_from_harness,harness_log_roots,final_error_type,final_signature,final_stage,final_success,final_source,harness_raw
0,astropy__astropy-12907,0,0,Qwen/Qwen2.5-Coder-7B-Instruct,1d006f36813d2fc095bc8da160e7e87abfd8d05ee8957d...,,,0,0,0,...,False,True,False,[/home/dibaeck/workspace/project_IR_sLM_MAS/ru...,GEN_FAIL,llm_timeout,GEN,False,pre_harness,NaN
1,astropy__astropy-14182,0,0,Qwen/Qwen2.5-Coder-7B-Instruct,323a3b727d26875c53e665ce8dcd10f792793ee5799a7a...,,,0,0,0,...,False,True,False,[/home/dibaeck/workspace/project_IR_sLM_MAS/ru...,GEN_FAIL,llm_timeout,GEN,False,pre_harness,NaN
2,astropy__astropy-14365,0,1,Qwen/Qwen2.5-Coder-7B-Instruct,232a841813d672f784b09b1e96ec3c80e68dec2004a896...,,"{\n ""edits"": [\n {\n ""op"": ""replace_r...",0,0,0,...,False,True,False,[/home/dibaeck/workspace/project_IR_sLM_MAS/ru...,APPLY_FAIL,edit_apply_path_missing,EDIT_APPLY,False,pre_harness,NaN


In [5]:
# 분석에 필요한 주요 컬럼 존재 여부 확인
cols_to_check = [
    "task_id",
    "instance_id",
    "attempt_index",
    "error_type",
    "stage",
    "signature",
    "success",
    "merged_from_harness",
    "final_error_type",
    "final_signature",
    "final_stage",
    "final_success",
    "policy_action",
]

existing = [c for c in cols_to_check if c in df.columns]
missing = [c for c in cols_to_check if c not in df.columns]

print("existing columns:", existing)
print("missing columns:", missing)

existing columns: ['task_id', 'instance_id', 'attempt_index', 'error_type', 'stage', 'signature', 'success', 'merged_from_harness', 'final_error_type', 'final_signature', 'final_stage', 'final_success', 'policy_action']
missing columns: []


In [6]:
# ===== 1) final_error_type 분포 =====
final_error_type_counts = (
    df["final_error_type"]
    .fillna("<<NA>>")
    .value_counts(dropna=False)
    .rename_axis("final_error_type")
    .reset_index(name="count")
)

display(final_error_type_counts)

,final_error_type,count
0,TEST_FAIL,76
1,APPLY_FAIL,17
2,GEN_FAIL,5
3,EDIT_PARSE_FAIL,2


In [7]:
# 비율까지 같이 보기
final_error_type_counts["ratio"] = final_error_type_counts["count"] / len(df)
display(final_error_type_counts)

,final_error_type,count,ratio
0,TEST_FAIL,76,0.76
1,APPLY_FAIL,17,0.17
2,GEN_FAIL,5,0.05
3,EDIT_PARSE_FAIL,2,0.02


In [8]:
# ===== 2) final_stage 분포 =====
final_stage_counts = (
    df["final_stage"]
    .fillna("<<NA>>")
    .value_counts(dropna=False)
    .rename_axis("final_stage")
    .reset_index(name="count")
)

final_stage_counts["ratio"] = final_stage_counts["count"] / len(df)
display(final_stage_counts)

,final_stage,count,ratio
0,TEST,76,0.76
1,EDIT_APPLY,17,0.17
2,GEN,5,0.05
3,EDIT_PARSE,2,0.02


In [9]:
# ===== 3) merged_from_harness 비율 =====
merged_counts = (
    df["merged_from_harness"]
    .fillna("<<NA>>")
    .value_counts(dropna=False)
    .rename_axis("merged_from_harness")
    .reset_index(name="count")
)

merged_counts["ratio"] = merged_counts["count"] / len(df)
display(merged_counts)

,merged_from_harness,count,ratio
0,True,76,0.76
1,False,24,0.24


In [10]:
# ===== 4) final_signature top-k =====
top_k = 20

final_signature_counts = (
    df["final_signature"]
    .fillna("<<NA>>")
    .value_counts(dropna=False)
    .head(top_k)
    .rename_axis("final_signature")
    .reset_index(name="count")
)

final_signature_counts["ratio"] = final_signature_counts["count"] / len(df)
display(final_signature_counts)

,final_signature,count,ratio
0,fail_to_pass_not_fixed,76,0.76
1,edit_apply_path_missing,15,0.15
2,llm_timeout,5,0.05
3,invalid_edit_script,2,0.02
4,edit_apply_range_oob,2,0.02


In [11]:
# ===== step2 후보 수 확인 =====
# 현재 exp2_step2 trigger 기준:
# merged_from_harness == True
# final_stage == "TEST"
# final_error_type == "TEST_FAIL"
step2_candidates = df[
    (df["merged_from_harness"] == True) &
    (df["final_stage"] == "TEST") &
    (df["final_error_type"] == "TEST_FAIL")
].copy()

print("step2 candidate count:", len(step2_candidates))
print("step2 candidate ratio:", len(step2_candidates) / len(df) if len(df) > 0 else 0.0)

display(
    step2_candidates[
        [
            c for c in [
                "task_id",
                "instance_id",
                "attempt_index",
                "policy_action",
                "final_error_type",
                "final_signature",
                "final_stage",
                "final_success",
                "merged_from_harness",
            ] if c in step2_candidates.columns
        ]
    ].head(20)
)

step2 candidate count: 76
step2 candidate ratio: 0.76


,task_id,instance_id,attempt_index,policy_action,final_error_type,final_signature,final_stage,final_success,merged_from_harness
3,astropy__astropy-14995,astropy__astropy-14995,1,RETRY_EXPAND_FILES,TEST_FAIL,fail_to_pass_not_fixed,TEST,False,True
5,astropy__astropy-7746,astropy__astropy-7746,0,INITIAL,TEST_FAIL,fail_to_pass_not_fixed,TEST,False,True
9,django__django-11019,django__django-11019,0,INITIAL,TEST_FAIL,fail_to_pass_not_fixed,TEST,False,True
11,django__django-11049,django__django-11049,0,INITIAL,TEST_FAIL,fail_to_pass_not_fixed,TEST,False,True
12,django__django-11099,django__django-11099,0,INITIAL,TEST_FAIL,fail_to_pass_not_fixed,TEST,False,True
13,django__django-11133,django__django-11133,0,INITIAL,TEST_FAIL,fail_to_pass_not_fixed,TEST,False,True
17,django__django-11564,django__django-11564,0,INITIAL,TEST_FAIL,fail_to_pass_not_fixed,TEST,False,True
18,django__django-11583,django__django-11583,0,INITIAL,TEST_FAIL,fail_to_pass_not_fixed,TEST,False,True
19,django__django-11620,django__django-11620,0,INITIAL,TEST_FAIL,fail_to_pass_not_fixed,TEST,False,True
20,django__django-11630,django__django-11630,0,INITIAL,TEST_FAIL,fail_to_pass_not_fixed,TEST,False,True


In [12]:
# step2 후보의 signature 분포
if len(step2_candidates) > 0:
    step2_sig_counts = (
        step2_candidates["final_signature"]
        .fillna("<<NA>>")
        .value_counts(dropna=False)
        .rename_axis("final_signature")
        .reset_index(name="count")
    )
    step2_sig_counts["ratio"] = step2_sig_counts["count"] / len(step2_candidates)
    display(step2_sig_counts)
else:
    print("No step2 candidates found.")

,final_signature,count,ratio
0,fail_to_pass_not_fixed,76,1.0


In [13]:
# 참고: policy_action별 final_error_type 교차표
# step1 정책이 최종적으로 어떤 결과로 이어졌는지 보는 용도
if "policy_action" in df.columns and "final_error_type" in df.columns:
    ctab = pd.crosstab(
        df["policy_action"].fillna("<<NA>>"),
        df["final_error_type"].fillna("<<NA>>"),
        margins=True
    )
    display(ctab)
else:
    print("policy_action or final_error_type column missing.")

final_error_type,APPLY_FAIL,EDIT_PARSE_FAIL,GEN_FAIL,TEST_FAIL,All
policy_action,,,,,
INITIAL,0,0,4,73,77
RETRY_EXPAND_FILES,17,0,1,2,20
RETRY_SCHEMA_CONSTRAINED,0,2,0,1,3
All,17,2,5,76,100


In [14]:
# 참고: initial PRED_READY였던 row만 따로 final outcome 보기
pred_ready_df = df[df["error_type"] == "PRED_READY"].copy()

print("initial PRED_READY rows:", len(pred_ready_df))

if len(pred_ready_df) > 0:
    pred_ready_final = (
        pred_ready_df["final_error_type"]
        .fillna("<<NA>>")
        .value_counts(dropna=False)
        .rename_axis("final_error_type")
        .reset_index(name="count")
    )
    pred_ready_final["ratio"] = pred_ready_final["count"] / len(pred_ready_df)
    display(pred_ready_final)

    pred_ready_stage = (
        pred_ready_df["final_stage"]
        .fillna("<<NA>>")
        .value_counts(dropna=False)
        .rename_axis("final_stage")
        .reset_index(name="count")
    )
    pred_ready_stage["ratio"] = pred_ready_stage["count"] / len(pred_ready_df)
    display(pred_ready_stage)

initial PRED_READY rows: 76


,final_error_type,count,ratio
0,TEST_FAIL,76,1.0


,final_stage,count,ratio
0,TEST,76,1.0


In [15]:
# step2 후보 샘플을 사람이 읽기 쉽게 보기
sample_cols = [
    "task_id",
    "instance_id",
    "repo",
    "attempt_index",
    "policy_action",
    "error_type",
    "signature",
    "final_error_type",
    "final_signature",
    "final_stage",
    "merged_from_harness",
    "final_success",
    "stderr",
    "exception",
]

sample_cols = [c for c in sample_cols if c in step2_candidates.columns]

display(step2_candidates[sample_cols].head(10))

,task_id,instance_id,repo,attempt_index,policy_action,error_type,signature,final_error_type,final_signature,final_stage,merged_from_harness,final_success,stderr,exception
3,astropy__astropy-14995,astropy__astropy-14995,astropy/astropy,1,RETRY_EXPAND_FILES,PRED_READY,ready_for_harness,TEST_FAIL,fail_to_pass_not_fixed,TEST,True,False,,
5,astropy__astropy-7746,astropy__astropy-7746,astropy/astropy,0,INITIAL,PRED_READY,ready_for_harness,TEST_FAIL,fail_to_pass_not_fixed,TEST,True,False,,
9,django__django-11019,django__django-11019,django/django,0,INITIAL,PRED_READY,ready_for_harness,TEST_FAIL,fail_to_pass_not_fixed,TEST,True,False,,
11,django__django-11049,django__django-11049,django/django,0,INITIAL,PRED_READY,ready_for_harness,TEST_FAIL,fail_to_pass_not_fixed,TEST,True,False,,
12,django__django-11099,django__django-11099,django/django,0,INITIAL,PRED_READY,ready_for_harness,TEST_FAIL,fail_to_pass_not_fixed,TEST,True,False,,
13,django__django-11133,django__django-11133,django/django,0,INITIAL,PRED_READY,ready_for_harness,TEST_FAIL,fail_to_pass_not_fixed,TEST,True,False,,
17,django__django-11564,django__django-11564,django/django,0,INITIAL,PRED_READY,ready_for_harness,TEST_FAIL,fail_to_pass_not_fixed,TEST,True,False,,
18,django__django-11583,django__django-11583,django/django,0,INITIAL,PRED_READY,ready_for_harness,TEST_FAIL,fail_to_pass_not_fixed,TEST,True,False,,
19,django__django-11620,django__django-11620,django/django,0,INITIAL,PRED_READY,ready_for_harness,TEST_FAIL,fail_to_pass_not_fixed,TEST,True,False,,
20,django__django-11630,django__django-11630,django/django,0,INITIAL,PRED_READY,ready_for_harness,TEST_FAIL,fail_to_pass_not_fixed,TEST,True,False,,


In [16]:
# 최종 요약 프린트
summary = {
    "total_rows": len(df),
    "initial_pred_ready": int((df["error_type"] == "PRED_READY").sum()) if "error_type" in df.columns else None,
    "merged_from_harness_true": int((df["merged_from_harness"] == True).sum()) if "merged_from_harness" in df.columns else None,
    "step2_candidates": len(step2_candidates),
}

summary

{'total_rows': 100,
 'initial_pred_ready': 76,
 'merged_from_harness_true': 76,
 'step2_candidates': 76}